# Speech Emotion Recognition with Wav2Vec2

Project-ready paths and setup are added to the supplied experiment. The analytical methods are preserved; this copy has no saved outputs. See `../originals/voice.ipynb` for the historical run and `../docs/REPRODUCIBILITY.md` for changes, dependencies, and limitations.

Run the setup cell first, then continue in order. Download datasets separately. Saved model weights are not included.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'data'
ARTIFACT_DIR = REPO_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
SPEECH_DATA_DIR = Path(os.environ.get('SPEECH_DATA_DIR', str(DATA_DIR / 'speech-emotion-recognition-en')))
TEXT_DATA_PATH = Path(os.environ.get('TEXT_DATA_PATH', str(DATA_DIR / 'text/tweet_emotions.csv')))
GLOVE_PATH = Path(os.environ.get('GLOVE_PATH', str(DATA_DIR / 'glove/glove.6B.50d.txt')))
SPEECH_SPLIT_DIR = ARTIFACT_DIR / 'speech_splits'
SPEECH_SPLIT_DIR.mkdir(exist_ok=True)

In [ ]:
# Install dependencies from the repository requirements files.


In [ ]:
import opendatasets as od
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torchaudio
import re
import difflib


In [ ]:
download_url = "https://www.kaggle.com/datasets/dmitrybabko/speech-emotion-recognition-en"
# Download/extract separately; see data/README.md.


In [ ]:
base_path = SPEECH_DATA_DIR


emotion_map = {
    "angry": "ANG",
    "disgust": "DIS",
    "fear": "FEA",
    "happy": "HAP",
    "neutral": "NEU",
    "sad": "SAD",
    "calm": "NEU",
    "surprise": "SUR",
}


def map_emotion_fuzzy(emotion_raw):
    closest = difflib.get_close_matches(emotion_raw.lower(), emotion_map.keys(), n=1)
    return emotion_map.get(closest[0]) if closest else None

def parse_crema(path):
    crema_data = []
    for wav_path in tqdm(list(Path(path).glob("*.wav")), desc="CREMA-D"):
        file_name = wav_path.name
        parts = file_name.split("_")
        if len(parts) == 4:
            speaker_id = parts[0]
            emotion = parts[2].lower()
            mapped_emotion = map_emotion_fuzzy(emotion)
            if mapped_emotion:
                crema_data.append({
                    "path": str(wav_path),
                    "emotion": mapped_emotion,
                    "speaker_id": speaker_id,
                    "dataset": "CREMA"
                })
    return pd.DataFrame(crema_data)

def parse_ravdess(path):
    ravdess_data = []
    for wav_path in tqdm(list(Path(path).rglob("*.wav")), desc="RAVDESS"):
        file_name = wav_path.name
        parts = file_name.split("-")
        if len(parts) >= 7:
            emotion_code = parts[2]
            speaker_id = parts[-1].split(".")[0]
            emotion_labels = {
                "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
                "05": "angry", "06": "fear", "07": "disgust", "08": "surprise"
            }
            raw_emotion = emotion_labels.get(emotion_code)
            mapped_emotion = map_emotion_fuzzy(raw_emotion)
            if mapped_emotion:
                ravdess_data.append({
                    "path": str(wav_path),
                    "emotion": mapped_emotion,
                    "speaker_id": speaker_id,
                    "dataset": "RAVDESS"
                })
    return pd.DataFrame(ravdess_data)

def parse_savee(path):
    savee_data = []
    emotion_map_savee = {
        "a": "ANG", "d": "DIS", "f": "FEA", "h": "HAP",
        "n": "NEU", "sa": "SAD", "su": "SUR"
    }

    for wav_path in tqdm(list(Path(path).glob("*.wav")), desc="SAVEE"):
        file_name = wav_path.stem.lower()


        match = re.search(r'_(sa|su|a|d|f|h|n)', file_name)
        if match:
            emotion_code = match.group(1)
            mapped_emotion = emotion_map_savee.get(emotion_code)
            if mapped_emotion:
                speaker_id = file_name[:2].upper()
                savee_data.append({
                    "path": str(wav_path),
                    "emotion": mapped_emotion,
                    "speaker_id": speaker_id,
                    "dataset": "SAVEE"
                })
    return pd.DataFrame(savee_data)

def parse_tess(path):
    tess_data = []
    for folder in tqdm(list(Path(path).glob("*")), desc="TESS"):
        if folder.is_dir():
            emotion_raw = folder.name.split("_")[-1].lower()
            mapped_emotion = map_emotion_fuzzy(emotion_raw)
            if mapped_emotion:
                for wav_path in folder.rglob("*.wav"):
                    speaker_id = wav_path.name.split("_")[0]
                    tess_data.append({
                        "path": str(wav_path),
                        "emotion": mapped_emotion,
                        "speaker_id": speaker_id,
                        "dataset": "TESS"
                    })
    return pd.DataFrame(tess_data)


df_crema = parse_crema(base_path / "Crema")
df_rav = parse_ravdess(base_path / "Ravdess")
df_savee = parse_savee(base_path / "Savee")
df_tess = parse_tess(base_path / "Tess")

df_all = pd.concat([df_crema, df_rav, df_savee, df_tess], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)


print(f"Total samples: {len(df_all)}")
print("Emotions:", df_all["emotion"].value_counts())
print("Datasets included:", df_all["dataset"].value_counts())


In [ ]:
# Split into train, validation and test
#train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["emotion"], random_state=42)
#val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["emotion"], random_state=42)

# Save as CSV
#os.makedirs("/content/data", exist_ok=True)
#train_df.to_csv(str(SPEECH_SPLIT_DIR / "train.csv"), sep="\t", index=False)
#val_df.to_csv(str(SPEECH_SPLIT_DIR / "validation.csv"), sep="\t", index=False)
#test_df.to_csv(str(SPEECH_SPLIT_DIR / "test.csv"), sep="\t", index=False)

#print(f"Train size: {len(train_df)}")
#print(f"Validation size: {len(val_df)}")
#print(f"Test size: {len(test_df)}")


In [ ]:
from sklearn.model_selection import train_test_split

# train / val / test
train_df, temp_df = train_test_split(df_all, test_size=0.3, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

# Save
SPEECH_SPLIT_DIR.mkdir(exist_ok=True)
train_df.to_csv(str(SPEECH_SPLIT_DIR / "train.csv"), sep="\t", index=False)
val_df.to_csv(str(SPEECH_SPLIT_DIR / "validation.csv"), sep="\t", index=False)
test_df.to_csv(str(SPEECH_SPLIT_DIR / "test.csv"), sep="\t", index=False)

# Report
def print_stats(name, df):
    print(f"\n{name} size: {len(df)}")
    print(f"{name} emotion distribution:\n{df['emotion'].value_counts()}")
    print(f"{name} dataset distribution:\n{df['dataset'].value_counts()}")

print_stats("Train", train_df)
print_stats("Validation", val_df)
print_stats("Test", test_df)


In [ ]:
# Load dataset with HuggingFace datasets
from datasets import load_dataset

data_files = {
    "train": str(SPEECH_SPLIT_DIR / "train.csv"),
    "validation": str(SPEECH_SPLIT_DIR / "validation.csv"),
    "test": str(SPEECH_SPLIT_DIR / "test.csv")
}

dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

# Check a sample
print(dataset["train"][0])


In [ ]:
# Set column names
input_column = "path"
output_column = "emotion"

# Get unique labels
label_list = dataset["train"].unique(output_column)
label_list.sort()
num_labels = len(label_list)
print(f"Labels: {label_list}")


In [ ]:
# Load pretrained feature extractor
from transformers import AutoConfig, Wav2Vec2Processor

model_name = "facebook/wav2vec2-base"
pooling_mode = "mean"

config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    label2id={label: i for i, label in enumerate(label_list)},
    id2label={i: label for i, label in enumerate(label_list)},
    finetuning_task="wav2vec2_clf",
)
setattr(config, 'pooling_mode', pooling_mode)

feature_extractor = Wav2Vec2Processor.from_pretrained(model_name)
target_sampling_rate = feature_extractor.feature_extractor.sampling_rate


In [ ]:
import librosa

# Updated Preprocessing with librosa
def speech_file_to_array_fn(path):
    speech, sr = librosa.load(path, sr=target_sampling_rate)  # Resample here directly
    return speech

def label_to_id(label):
    return label_list.index(label)

def preprocess_function(batch):
    speech = speech_file_to_array_fn(batch[input_column])
    batch["input_values"] = feature_extractor(speech, sampling_rate=target_sampling_rate).input_values[0]
    batch["labels"] = label_to_id(batch[output_column])
    return batch

dataset = dataset.map(preprocess_function)
print(dataset)


In [ ]:
# Define model
import torch
import torch.nn as nn
from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Model, Wav2Vec2PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput

class Wav2Vec2ForSpeechClassification(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.pooling_mode = config.pooling_mode

        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(config.hidden_size, config.num_labels),
        )


        self.init_weights()
        #for param in self.wav2vec2.parameters():
           # param.requires_grad = False

    def forward(self, input_values, attention_mask=None, labels=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden_states = outputs[0]

        if self.pooling_mode == "mean":
            hidden_states = hidden_states.mean(dim=1)
        elif self.pooling_mode == "sum":
            hidden_states = hidden_states.sum(dim=1)
        elif self.pooling_mode == "max":
            hidden_states = hidden_states.max(dim=1)[0]
        else:
            raise ValueError(f"Invalid pooling mode: {self.pooling_mode}")

        logits = self.classifier(hidden_states)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


In [ ]:
# Initialize model
model = Wav2Vec2ForSpeechClassification.from_pretrained(model_name, config=config)


In [ ]:
# Define data collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollator:
    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Union[List[float], int]]]) -> Dict[str, torch.Tensor]:
        input_values = [f["input_values"] for f in features]
        labels = [f["labels"] for f in features]
        batch = self.feature_extractor.pad(
            {"input_values": input_values},
            return_tensors="pt"
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm import tqdm
import numpy as np

Batch = 16


data_collator = DataCollator(feature_extractor)

train_loader = DataLoader(dataset["train"], batch_size=Batch, shuffle=True, collate_fn=data_collator)
eval_loader = DataLoader(dataset["validation"], batch_size=Batch, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(dataset["test"], batch_size=Batch, shuffle=False, collate_fn=data_collator)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
# Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_training_steps = len(train_loader) * 20  # num_epochs
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * total_training_steps),
    num_training_steps=total_training_steps,
)


In [ ]:
# Early stopping config
early_stopping_patience = 6
best_val_loss = float("inf")
epochs_no_improve = 0

# Evaluation function
def evaluate_full(model, dataloader):
    model.eval()
    eval_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch in dataloader:
            input_values = batch["input_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_values=input_values, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            eval_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return eval_loss / len(dataloader), correct / total


In [ ]:
#  Train loop
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_values=input_values, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  #gradient clipping
        optimizer.step()
        lr_scheduler.step()

        train_loss += loss.item()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_acc = correct / total
    val_loss, val_acc = evaluate_full(model, eval_loader)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), str(ARTIFACT_DIR / "speech_best_model.pt"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stopping_patience:
            print(" Early stopping triggered.")
            break


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def test_evaluation(model, dataloader, label_names, device):
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_values = batch["input_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_values=input_values)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")

    print("\n Classification Report:\n", classification_report(all_labels, all_preds, target_names=label_names))

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix - Test Set")
    plt.tight_layout()
    plt.show()


In [ ]:
label_names = list(config.id2label.values())

test_evaluation(model, test_loader, label_names, device)
